# Maintainers Copilot — OpenAI LLM Baseline

This is the third classifier track. It uses OpenAI through the Responses API with Structured Outputs to classify the same balanced comparison examples as the DistilBERT and classical baselines.

No GPU is needed. You need an `OPENAI_API_KEY`.


## 1. Install dependencies

In [ ]:
!pip -q install openai scikit-learn

## 2. Mount Drive and configure the run

This notebook uses `test_200_balanced.jsonl` by default. Start with `FINAL_RUN = False` for a 50-example smoke test. For the fair three-way comparison, set `FINAL_RUN = True`; that evaluates all 200 balanced examples.

Requests are batched with `BATCH_SIZE = 20`, so the 200-example comparison uses about 10 OpenAI calls instead of 200.


In [ ]:
from __future__ import annotations

import getpass
import hashlib
import json
import os
from collections import Counter
from pathlib import Path
from time import perf_counter, sleep
from typing import Any

from google.colab import drive, userdata
from openai import OpenAI
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

drive.mount('/content/drive')

api_key = userdata.get('OPENAI_API_KEY')
if not api_key:
    api_key = getpass.getpass('Enter OPENAI_API_KEY: ')
client = OpenAI(api_key=api_key)

DRIVE_ROOT = Path('/content/drive/MyDrive/maintainers-copilot')
DATA_DIR = DRIVE_ROOT / 'data'
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'

MODEL = 'gpt-4o-mini'
FINAL_RUN = False  # False = 50-example smoke test. True = full 200-example comparison subset.
LIMIT = None if FINAL_RUN else 50
BATCH_SIZE = 20
MAX_BODY_CHARS = 2000
TEST_FILENAME = 'test_200_balanced.jsonl'
RUN_NAME = 'openai-gpt-4o-mini-test-200' if FINAL_RUN else 'openai-gpt-4o-mini-test-200-pilot'
RUN_DIR = ARTIFACT_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

TEST_PATH = DATA_DIR / TEST_FILENAME
if not TEST_PATH.exists():
    raise FileNotFoundError(f'Missing comparison test split: {TEST_PATH}')

print('Model:', MODEL)
print('Test path:', TEST_PATH)
print('Limit:', LIMIT)
print('Batch size:', BATCH_SIZE)
print('Run dir:', RUN_DIR)


## 3. Helpers and prompt schema

In [ ]:
TARGET_LABELS = ('bug', 'feature', 'docs', 'question')
LABEL_TO_ID = {'bug': 0, 'feature': 1, 'docs': 2, 'question': 3}
INPUT_PRICE_PER_MTOK = 0.15
OUTPUT_PRICE_PER_MTOK = 0.60


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    with path.open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            if not isinstance(record, dict):
                raise ValueError(f'Expected an object on line {line_number} of {path}.')
            records.append(record)
    return records


def fingerprint_jsonl(path: Path) -> dict[str, Any]:
    digest = hashlib.sha256()
    examples = 0
    with path.open('rb') as handle:
        for line in handle:
            if line.strip():
                examples += 1
            digest.update(line)
    return {'path': str(path), 'sha256': digest.hexdigest(), 'examples': examples}


def system_prompt() -> str:
    return """You are a maintainer triage classifier for GitHub issues.

Task:
Choose exactly one primary label for the issue using only the title and body.
Do not use repository labels, comments, metadata, author identity, or outside knowledge.

Allowed labels:
- bug: a defect, regression, crash, incorrect result, failing test, broken existing behavior, or compatibility problem.
- feature: a request for a new capability, enhancement, API addition/change, performance improvement, or behavior improvement that is not primarily a defect.
- docs: documentation, examples, tutorials, website/reference text, docstrings, wording, typo, or explanation improvements.
- question: usage help, troubleshooting, clarification, installation/environment help, or “how do I?” requests.

Tie-breaking rules:
1. If the issue reports current behavior that appears wrong or broken, choose bug even if the fix may require code changes.
2. If the issue asks to add or change behavior but does not show existing behavior is broken, choose feature.
3. If the requested change is mainly to docs/examples/wording, choose docs.
4. If the author mostly asks for help understanding or using the project, choose question.
5. If multiple labels seem plausible, choose the label a maintainer would most likely use as the primary triage label.

Return only the structured JSON object requested by the caller. The label value must be exactly one of: bug, feature, docs, question."""


def build_batch_user_prompt(indexed_records: list[tuple[int, dict[str, Any]]], max_body_chars: int = 2000) -> str:
    items = []
    for index, record in indexed_records:
        title = (record.get('title') or '').strip()
        body = (record.get('body') or '').strip()[:max_body_chars]
        items.append(f'ID: {index}\nTitle: {title}\nBody: {body}')
    return (
        'Classify each GitHub issue below. Return exactly one prediction per ID. '
        'Do not use hidden labels; only infer from title/body.\n\n'
        + '\n\n---\n\n'.join(items)
    )

def batch_classification_schema() -> dict[str, Any]:
    return {
        'type': 'object',
        'properties': {
            'predictions': {
                'type': 'array',
                'items': {
                    'type': 'object',
                    'properties': {
                        'id': {'type': 'integer'},
                        'label': {'type': 'string', 'enum': list(TARGET_LABELS)},
                        'confidence': {'type': 'number'},
                        'rationale': {'type': 'string'},
                    },
                    'required': ['id', 'label', 'confidence', 'rationale'],
                    'additionalProperties': False,
                },
            }
        },
        'required': ['predictions'],
        'additionalProperties': False,
    }


def usage_to_dict(usage: Any) -> dict[str, int]:
    if usage is None:
        return {}
    result: dict[str, int] = {}
    for key in ('input_tokens', 'output_tokens', 'total_tokens'):
        value = getattr(usage, key, None)
        if isinstance(value, int):
            result[key] = value
    return result


def load_existing_predictions(path: Path) -> dict[int, dict[str, Any]]:
    if not path.exists():
        return {}
    existing = {}
    for line in path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        record = json.loads(line)
        existing[int(record['index'])] = record
    return existing


def append_jsonl(path: Path, record: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, ensure_ascii=False, sort_keys=True))
        handle.write('\n')


## 4. Run predictions

This cell writes `predictions.jsonl` incrementally, so if Colab disconnects you can rerun with the same `RUN_NAME` and it will resume completed examples. Each OpenAI request contains up to `BATCH_SIZE` issues.


In [ ]:
records = read_jsonl(TEST_PATH)
if LIMIT is not None:
    records = records[:LIMIT]

prediction_path = RUN_DIR / 'predictions.jsonl'
predictions_by_index = load_existing_predictions(prediction_path)

for batch_start in range(0, len(records), BATCH_SIZE):
    batch = [
        (index, records[index])
        for index in range(batch_start, min(batch_start + BATCH_SIZE, len(records)))
        if index not in predictions_by_index
    ]
    if not batch:
        continue

    request = {
        'model': MODEL,
        'instructions': system_prompt(),
        'input': build_batch_user_prompt(batch, max_body_chars=MAX_BODY_CHARS),
        'text': {
            'format': {
                'type': 'json_schema',
                'name': 'issue_classification_batch',
                'strict': True,
                'schema': batch_classification_schema(),
            }
        },
    }
    started = perf_counter()
    response = client.responses.create(**request)
    latency_seconds = perf_counter() - started
    parsed = json.loads(response.output_text)

    by_id = {int(item['id']): item for item in parsed['predictions']}
    expected_ids = {index for index, _ in batch}
    if set(by_id) != expected_ids:
        raise ValueError(f'Batch response IDs did not match request IDs: expected {expected_ids}, got {set(by_id)}')

    batch_usage = usage_to_dict(getattr(response, 'usage', None))
    for offset, (index, record) in enumerate(batch):
        item = by_id[index]
        label = item['label']
        if label not in LABEL_TO_ID:
            raise ValueError(f'Unsupported label from model: {label!r}')
        prediction = {
            'index': index,
            'id': record.get('id'),
            'target': record['target'],
            'prediction': label,
            'confidence': item.get('confidence'),
            'rationale': item.get('rationale'),
            'latency_seconds': latency_seconds / len(batch),
            'batch_size': len(batch),
            'usage': batch_usage if offset == 0 else {},
            'usage_attribution': 'first_record_in_batch' if offset == 0 else 'counted_on_first_record_in_batch',
            'response_id': getattr(response, 'id', None),
        }
        append_jsonl(prediction_path, prediction)
        predictions_by_index[index] = prediction

    print(f'Completed {min(batch_start + BATCH_SIZE, len(records))}/{len(records)}')

print('Predictions:', len(predictions_by_index), 'of', len(records))


## 5. Compute metrics and save evidence

In [ ]:
ordered = [predictions_by_index[index] for index in range(len(records))]
actual = [record['target'] for record in records]
predicted = [record['prediction'] for record in ordered]

report = classification_report(
    actual,
    predicted,
    labels=list(TARGET_LABELS),
    output_dict=True,
    zero_division=0,
)
usage = Counter()
for record in ordered:
    for key, value in (record.get('usage') or {}).items():
        if isinstance(value, int):
            usage[key] += value
latency_total = sum(float(record.get('latency_seconds', 0.0)) for record in ordered)
estimated_cost = None
if 'input_tokens' in usage and 'output_tokens' in usage:
    estimated_cost = (usage['input_tokens'] / 1_000_000 * INPUT_PRICE_PER_MTOK) + (
        usage['output_tokens'] / 1_000_000 * OUTPUT_PRICE_PER_MTOK
    )

metrics = {
    'split': 'test_200_balanced' if TEST_FILENAME == 'test_200_balanced.jsonl' else 'test',
    'examples': len(records),
    'accuracy': float(accuracy_score(actual, predicted)),
    'macro_f1': float(f1_score(actual, predicted, labels=list(TARGET_LABELS), average='macro')),
    'weighted_f1': float(f1_score(actual, predicted, labels=list(TARGET_LABELS), average='weighted')),
    'per_class_f1': {label: float(report[label]['f1-score']) for label in TARGET_LABELS},
    'confusion_matrix_labels': list(TARGET_LABELS),
    'confusion_matrix': confusion_matrix(actual, predicted, labels=list(TARGET_LABELS)).tolist(),
    'latency': {
        'total_seconds': latency_total,
        'mean_seconds_per_example': latency_total / len(records) if records else None,
        'examples_per_second': len(records) / latency_total if latency_total else None,
    },
    'usage': dict(usage),
    'estimated_cost_usd': estimated_cost,
}
manifest = {
    'experiment': 'openai_llm_baseline',
    'config': {
        'run_name': RUN_NAME,
        'provider': 'openai',
        'model': MODEL,
        'limit': LIMIT,
        'batch_size': BATCH_SIZE,
        'max_body_chars': MAX_BODY_CHARS,
        'input_price_per_mtok': INPUT_PRICE_PER_MTOK,
        'output_price_per_mtok': OUTPUT_PRICE_PER_MTOK,
    },
    'labels': LABEL_TO_ID,
    'dataset': {'test': fingerprint_jsonl(TEST_PATH)},
    'prompt': {
        'system': system_prompt(),
        'schema': batch_classification_schema(),
        'text_policy': 'batched issue IDs plus title and truncated body; no repository labels shown to model',
    },
    'docs': {
        'responses_api': 'https://developers.openai.com/api/docs/guides/text',
        'structured_outputs': 'https://developers.openai.com/api/docs/guides/structured-outputs',
        'models': 'https://developers.openai.com/api/docs/models',
    },
}

(RUN_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8')
(RUN_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2, sort_keys=True) + '\n', encoding='utf-8')
(RUN_DIR / 'classification_report.json').write_text(json.dumps({metrics['split']: report}, indent=2, sort_keys=True) + '\n', encoding='utf-8')

print('Wrote evidence to:', RUN_DIR)
metrics


## 6. Files to bring back to the repo

For the final 200-example comparison run, copy these files into `model_server/classifier/runs/openai-gpt-4o-mini-test-200/`:

```text
run_manifest.json
metrics.json
classification_report.json
```

`predictions.jsonl` is useful for debugging and resumability. Commit it only if it remains small enough and does not make review noisy.
